In [ ]:
# -*- coding: utf-8 -*-
"""
只测试 @Reuters 的推文抓取情况：
- 看这个 api.apidance.pro/sapi/UserTweets 实际能返回的时间范围
- 不做时间过滤，只统计最早 / 最晚的 created_at
"""

import json
import http.client
import time
import logging
from datetime import datetime

# =============== 配置 ===============
API_HOST = "api.apidance.pro"
API_KEY = "q4fa83ok43io70najdgmijdt2s6fkl"

TARGET_SCREEN_NAME = "Reuters"
MAX_PAGES = 1000  # 最多翻 10 页，够我们看时间范围了

# 你想要的时间下限，用来对比（不会在代码里过滤，只是打印一下）
START_DATE_TARGET = datetime(2024, 1, 1)

# 日志简单配置
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)


# =============== 基础请求函数 ===============

def get_user_rest_id(user_name: str):
    """通过 screen_name 获取用户 rest_id"""
    try:
        conn = http.client.HTTPSConnection(API_HOST)
        headers = {'apikey': API_KEY}

        url = (
            "/graphql/UserByScreenName?"
            "variables=%7B%22screen_name%22:%22" + user_name +
            "%22,%22withSafetyModeUserFields%22:true,%22withHighlightedLabel%22:true%7D"
        )

        conn.request("GET", url, '', headers)
        res = conn.getresponse()
        data = res.read()
        json_data = json.loads(data.decode("utf-8"))

        rest_id = json_data['data']['user']['result']['rest_id']
        logger.info(f"✓ 获取用户 @{user_name} 的 rest_id: {rest_id}")
        return rest_id

    except Exception as e:
        logger.error(f"✗ 获取用户 @{user_name} 的 rest_id 失败: {e}")
        return None


def get_user_tweets_page(user_id: str, cursor: str = None):
    """获取单页用户推文（按 API 原样返回）"""
    conn = http.client.HTTPSConnection(API_HOST)
    headers = {'apikey': API_KEY}

    url = f"/sapi/UserTweets?user_id={user_id}"
    if cursor:
        url += f"&cursor={cursor}"
    else:
        url += "&cursor=null"

    conn.request("GET", url, '', headers)
    res = conn.getresponse()
    data = res.read()
    return json.loads(data.decode("utf-8"))


# =============== 工具函数 ===============

def parse_tweet_datetime(tweet: dict):
    """
    按你给的样例：
    "created_at": "Sat Dec 06 11:50:19 +0000 2025"
    """
    created_at_str = tweet.get("created_at")
    if not created_at_str:
        return None

    try:
        dt = datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.replace(tzinfo=None)  # 去掉时区，方便比较
    except Exception as e:
        logger.warning(f"解析 created_at 失败: {created_at_str} ({e})")
        return None


# =============== 核心测试逻辑 ===============

def test_reuters(max_pages: int = 10):
    user_name = TARGET_SCREEN_NAME
    user_id = get_user_rest_id(user_name)
    if not user_id:
        logger.error("无法获取 @Reuters rest_id，直接退出")
        return

    logger.info(f"开始测试 @ {user_name} 推文，最多翻 {max_pages} 页")

    all_tweets = []
    cursor = None
    page = 1

    min_dt = None
    max_dt = None

    while page <= max_pages:
        logger.info(f"--- 第 {page} 页 ---")
        try:
            resp = get_user_tweets_page(user_id, cursor)
            tweets = resp.get("tweets", [])

            if not tweets:
                logger.info("这一页没有 tweets，API 可能不给更多了，结束。")
                break

            logger.info(f"本页 tweets 数量：{len(tweets)}")

            for tw in tweets:
                dt = parse_tweet_datetime(tw)
                if dt is None:
                    continue

                if (min_dt is None) or (dt < min_dt):
                    min_dt = dt
                if (max_dt is None) or (dt > max_dt):
                    max_dt = dt

                all_tweets.append(tw)

            logger.info(f"当前已统计 tweets 数：{len(all_tweets)}")
            logger.info(f"当前时间范围大致为：[{min_dt}  ~  {max_dt}]")

            next_cursor = resp.get("next_cursor_str")
            if not next_cursor:
                logger.info("没有 next_cursor 了，API 不再给更多历史。")
                break

            cursor = next_cursor
            page += 1
            time.sleep(0.3)

        except Exception as e:
            logger.error(f"获取第 {page} 页时出错：{e}")
            break

    logger.info("===== 测试结束 =====")
    logger.info(f"总共拿到 tweets 数：{len(all_tweets)}")
    logger.info(f"最早一条时间：{min_dt}")
    logger.info(f"最晚一条时间：{max_dt}")
    logger.info(f"你设想的时间下限是：{START_DATE_TARGET}")

    # 保存一下原始数据，方便你手动 inspect
    if all_tweets:
        out_file = "reuters_raw.json"
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(all_tweets, f, indent=2, ensure_ascii=False)
        logger.info(f"已将 tweets 原始数据保存到：{out_file}")


if __name__ == "__main__":
    test_reuters(MAX_PAGES)


In [ ]:
# -*- coding: utf-8 -*-
"""
只测试 @Reuters 的推文抓取情况（更耐艹版）：
- 每页之间加延迟 + 随机抖动
- 如果某一页没返回 tweets，会在这一页重试多次
  多次失败后，才认为真的到尽头
"""

import json
import http.client
import time
import logging
import random
from datetime import datetime

API_HOST = "api.apidance.pro"
API_KEY = "q4fa83ok43io70najdgmijdt2s6fkl"

TARGET_SCREEN_NAME = "Reuters"

MAX_PAGES_HARD = 500          # 最多尝试翻 500 页，防止无限循环
MAX_RETRIES_PER_PAGE = 3      # 每一页最多重试次数
BASE_SLEEP_BETWEEN_PAGES = 0.8   # 每一页之间基础延迟（秒）
EXTRA_JITTER = 0.6            # 额外随机抖动（秒），防止节奏太规律

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)


def get_user_rest_id(user_name: str):
    """通过 screen_name 获取用户 rest_id"""
    try:
        conn = http.client.HTTPSConnection(API_HOST)
        headers = {'apikey': API_KEY}
        url = (
            "/graphql/UserByScreenName?"
            "variables=%7B%22screen_name%22:%22" + user_name +
            "%22,%22withSafetyModeUserFields%22:true,%22withHighlightedLabel%22:true%7D"
        )
        conn.request("GET", url, '', headers)
        res = conn.getresponse()
        data = res.read()
        json_data = json.loads(data.decode("utf-8"))
        rest_id = json_data['data']['user']['result']['rest_id']
        logger.info(f"✓ 获取用户 @{user_name} 的 rest_id: {rest_id}")
        return rest_id
    except Exception as e:
        logger.error(f"✗ 获取用户 @{user_name} 的 rest_id 失败: {e}")
        return None


def get_user_tweets_page(user_id: str, cursor: str = None):
    """获取单页用户推文（按 API 原样返回，一次请求）"""
    conn = http.client.HTTPSConnection(API_HOST)
    headers = {'apikey': API_KEY}
    url = f"/sapi/UserTweets?user_id={user_id}"
    if cursor:
        url += f"&cursor={cursor}"
    else:
        url += "&cursor=null"
    conn.request("GET", url, '', headers)
    res = conn.getresponse()
    data = res.read()
    return json.loads(data.decode("utf-8"))


def parse_tweet_datetime(tweet: dict):
    """
    按你给的样例：
    "created_at": "Sat Dec 06 11:50:19 +0000 2025"
    """
    created_at_str = tweet.get("created_at")
    if not created_at_str:
        return None
    try:
        dt = datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.replace(tzinfo=None)
    except Exception as e:
        logger.warning(f"解析 created_at 失败: {created_at_str} ({e})")
        return None


def test_reuters_window_robust():
    user_name = TARGET_SCREEN_NAME
    user_id = get_user_rest_id(user_name)
    if not user_id:
        logger.error("无法获取 @Reuters rest_id，直接退出")
        return

    logger.info(
        f"开始【加延迟+重试】测试 @{user_name} 的可见时间窗口 "
        f"(最多 {MAX_PAGES_HARD} 页，每页最多重试 {MAX_RETRIES_PER_PAGE} 次)"
    )

    all_tweets = []
    cursor = None
    page = 1

    min_dt = None
    max_dt = None

    while page <= MAX_PAGES_HARD:
        logger.info(f"=== 第 {page} 页 ===")

        tweets = None
        resp = None

        # ------------------ 这一页内部重试 ------------------
        for attempt in range(1, MAX_RETRIES_PER_PAGE + 1):
            try:
                logger.info(f"请求第 {page} 页，第 {attempt} 次尝试...")
                resp = get_user_tweets_page(user_id, cursor)
                tweets = resp.get("tweets")

                if tweets is None:
                    logger.warning(
                        f"第 {page} 页，第 {attempt} 次：响应中没有 'tweets' 字段，"
                        "可能是异常结构，准备重试..."
                    )
                elif len(tweets) == 0:
                    logger.warning(
                        f"第 {page} 页，第 {attempt} 次：tweets 数量为 0，"
                        "可能触发限流 / 窗口尽头，准备重试..."
                    )
                else:
                    # 正常拿到 tweets，结束这一页的重试
                    break

            except Exception as e:
                logger.error(
                    f"第 {page} 页，第 {attempt} 次请求出错：{e}，准备稍后重试..."
                )

            # 只有在失败的情况下才执行这里的 sleep
            sleep_sec = BASE_SLEEP_BETWEEN_PAGES * attempt + random.uniform(0, EXTRA_JITTER)
            logger.info(f"第 {page} 页，第 {attempt} 次失败后，睡眠 {sleep_sec:.2f} 秒再试")
            time.sleep(sleep_sec)

        # ------------------ 重试结束后的判断 ------------------
        if not tweets:
            logger.info(
                f"第 {page} 页在重试 {MAX_RETRIES_PER_PAGE} 次后仍然没有有效 tweets，"
                "认为已经达到 API 可见历史尽头或严重限流，停止。"
            )
            break

        # 正常处理这一页的 tweets
        logger.info(f"第 {page} 页最终拿到 tweets 数量：{len(tweets)}")

        for tw in tweets:
            dt = parse_tweet_datetime(tw)
            if dt is None:
                continue

            if (min_dt is None) or (dt < min_dt):
                min_dt = dt
            if (max_dt is None) or (dt > max_dt):
                max_dt = dt

            all_tweets.append(tw)

        logger.info(f"当前已统计 tweets 数：{len(all_tweets)}")
        logger.info(f"当前时间范围大致为：[{min_dt}  ~  {max_dt}]")

        # 处理 next_cursor
        next_cursor = resp.get("next_cursor_str") if isinstance(resp, dict) else None
        if not next_cursor:
            logger.info("没有 next_cursor 了，API 不再给更多历史。")
            break

        cursor = next_cursor
        page += 1

        # 页与页之间也加一点延迟 + 抖动
        sleep_sec = BASE_SLEEP_BETWEEN_PAGES + random.uniform(0, EXTRA_JITTER)
        logger.info(f"页与页之间睡眠 {sleep_sec:.2f} 秒，防止请求过于频繁")
        time.sleep(sleep_sec)

    logger.info("===== 测试结束 =====")
    logger.info(f"总共拿到 tweets 数：{len(all_tweets)}")
    logger.info(f"最早一条时间：{min_dt}")
    logger.info(f"最晚一条时间：{max_dt}")

    if all_tweets:
        out_file = "reuters_window_robust.json"
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(all_tweets, f, indent=2, ensure_ascii=False)
        logger.info(f"已将 tweets 原始数据保存到：{out_file}")


if __name__ == "__main__":
    test_reuters_window_robust()


In [ ]:
import http.client

conn = http.client.HTTPSConnection("api.apidance.pro")
payload = ''
headers = {
   'apikey': 'q4fa83ok43io70najdgmijdt2s6fkl'
}
conn.request("GET", "/graphql/SearchTimeline?variables=%7B%22rawQuery%22:%22twitter%22,%22count%22:40,%22cursor%22:%22%22,%22querySource%22:%22typed_query%22,%22product%22:%22Latest%22,%22includePromotedContent%22:false%7D", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

In [ ]:
# -*- coding: utf-8 -*-
"""
测试 apidance 的 GraphQL SearchTimeline：
- rawQuery = "from:Reuters since:2025-12-01 until:2025-12-07"
- 解析 Tweet 和下一页 cursor
"""

import json
import http.client
import time
import logging
from urllib.parse import quote
from datetime import datetime

API_HOST = "api.apidance.pro"
API_KEY = "q4fa83ok43io70najdgmijdt2s6fkl"   # TODO: 换成你自己的

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)


def call_searchtimeline(raw_query: str, cursor: str | None):
    """调用 /graphql/SearchTimeline"""
    conn = http.client.HTTPSConnection(API_HOST)
    headers = {"apikey": API_KEY}

    variables = {
        "rawQuery": raw_query,
        "count": 40,                  # 一页 40 条
        "cursor": cursor or "",
        "querySource": "typed_query",
        "product": "Latest",
        "includePromotedContent": False
    }
    variables_str = json.dumps(variables, separators=(",", ":"))
    encoded_vars = quote(variables_str, safe="")

    path = f"/graphql/SearchTimeline?variables={encoded_vars}"

    conn.request("GET", path, "", headers)
    res = conn.getresponse()
    text = res.read().decode("utf-8")

    try:
        js = json.loads(text)
    except Exception as e:
        logger.error(f"解析 JSON 失败: {e}, 原始片段: {text[:200]}")
        raise

    return js


def parse_tweet_datetime(created_at_str: str | None):
    if not created_at_str:
        return None
    try:
        dt = datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.replace(tzinfo=None)
    except Exception as e:
        logger.warning(f"解析 created_at 失败: {created_at_str} ({e})")
        return None


def extract_tweets_and_cursor(resp: dict):
    """
    按你贴的 SearchTimeline 返回结构解析：
    - tweet: content.__typename == "TimelineTimelineItem" -> itemContent.tweet_results.result.legacy
    - cursor: content.__typename == "TimelineTimelineCursor" 且 cursorType == "Bottom"
    """
    tweets = []
    next_cursor = None

    data = resp.get("data") or {}
    search_obj = data.get("search_by_raw_query") or data.get("searchByRawQuery") or {}
    search_timeline = search_obj.get("search_timeline") or search_obj.get("searchTimeline") or {}
    timeline = search_timeline.get("timeline") or {}
    instructions = timeline.get("instructions") or []

    for instr in instructions:
        entries = instr.get("entries") or []
        for entry in entries:
            content = entry.get("content") or {}
            ctype = content.get("__typename")

            # 1) tweet item
            if ctype == "TimelineTimelineItem":
                item_content = content.get("itemContent") or {}
                if item_content.get("__typename") != "TimelineTweet":
                    continue

                tweet_results = item_content.get("tweet_results") or item_content.get("tweetResults") or {}
                result = tweet_results.get("result") or {}
                legacy = result.get("legacy") or {}

                if not legacy:
                    continue

                tid = legacy.get("id_str") or legacy.get("id")
                text = legacy.get("full_text") or legacy.get("text")
                created_at = legacy.get("created_at")

                tweets.append({
                    "tweet_id": tid,
                    "text": text,
                    "created_at": created_at,
                    "legacy": legacy
                })

            # 2) cursor entry
            elif ctype == "TimelineTimelineCursor":
                if content.get("cursorType") == "Bottom":
                    value = content.get("value")
                    if value:
                        next_cursor = value

    return tweets, next_cursor


def test_one_range():
    raw_query = "from:Reuters since:2025-12-01 until:2025-12-07"
    logger.info(f"测试 rawQuery: {raw_query}")

    cursor = None
    page = 1
    all_tweets = []

    while page <= 5:  # 先试 5 页够不够
        logger.info(f"--- 第 {page} 页 ---")
        resp = call_searchtimeline(raw_query, cursor)
        tweets, next_cursor = extract_tweets_and_cursor(resp)

        logger.info(f"本页解析出 tweets 数量: {len(tweets)}")
        all_tweets.extend(tweets)

        logger.info(f"当前累计 tweets: {len(all_tweets)}")

        if not next_cursor:
            logger.info("没有 next_cursor，结束")
            break

        cursor = next_cursor
        page += 1
        time.sleep(0.5)

    # 看一下时间范围
    dts = [parse_tweet_datetime(t["created_at"]) for t in all_tweets if t.get("created_at")]
    dts = [d for d in dts if d]
    if dts:
        logger.info(f"拿到的时间范围大致为: [{min(dts)} ~ {max(dts)}]")

    # 简单保存一下看看结构
    with open("reuters_search_test.json", "w", encoding="utf-8") as f:
        json.dump(all_tweets, f, indent=2, ensure_ascii=False)
    logger.info("测试数据已保存到 reuters_search_test.json")


if __name__ == "__main__":
    test_one_range()


In [ ]:
# -*- coding: utf-8 -*-
"""
使用 /sapi/UserTweets + 你的耐艹版框架
对 a_kol.xlsx 里的所有 KOL：
- 每个账号用 UserTweets 一页一页往前翻
- 每页内部多次重试
- 统计每个账号：
    - 总 tweets 数（本次跑到的）
    - 最早一条 tweet 时间
    - 最晚一条 tweet 时间
    - 是否已经翻到 2024-01-01 及以前
- 输出一个 Excel：kol_usertweets_window.xlsx
"""

import json
import http.client
import time
import logging
import random
from datetime import datetime
from typing import Optional, Dict, Any, List

import pandas as pd
import os
import traceback

# ==================== 全局配置 ====================

API_HOST = "api.apidance.pro"
API_KEY = "q4fa83ok43io70najdgmijdt2s6fkl"

MAX_PAGES_HARD = 500            # 每个用户最多翻 500 页（上限，防止死循环）
MAX_RETRIES_PER_PAGE = 10      # 每一页最多重试次数
BASE_SLEEP_BETWEEN_PAGES = 1.8  # 页与页之间基础延迟（秒）
EXTRA_JITTER = 0.6              # 额外随机抖动（秒）

COVER_TARGET_DATE = datetime(2024, 1, 1)  # 判断是否覆盖到 2024-01-01

# 文件路径（可以按你自己的路径改）
KOL_EXCEL_PATH = r"I:\finance-agent\X\a_kol.xlsx"
OUT_SUMMARY_PATH = r"I:\finance-agent\X\kol_usertweets_window.xlsx"
LOG_DIR = r"I:\finance-agent\X\logs"

os.makedirs(LOG_DIR, exist_ok=True)
log_filename = os.path.join(
    LOG_DIR, f"kol_usertweets_window_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_filename, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


# ==================== 工具函数 ====================

def get_user_rest_id(user_name: str, max_retries: int = 3) -> Optional[str]:
    """通过 screen_name 获取用户 rest_id，带简单重试。"""
    for attempt in range(1, max_retries + 1):
        try:
            conn = http.client.HTTPSConnection(API_HOST, timeout=30)
            headers = {'apikey': API_KEY}
            url = (
                "/graphql/UserByScreenName?"
                "variables=%7B%22screen_name%22:%22" + user_name +
                "%22,%22withSafetyModeUserFields%22:true,%22withHighlightedLabel%22:true%7D"
            )
            conn.request("GET", url, '', headers)
            res = conn.getresponse()
            data = res.read()
            json_data = json.loads(data.decode("utf-8"))
            rest_id = json_data['data']['user']['result']['rest_id']
            logger.info(f"✓ 获取用户 @{user_name} 的 rest_id: {rest_id}")
            return rest_id
        except Exception as e:
            logger.error(f"✗ 获取用户 @{user_name} 的 rest_id 失败 (第 {attempt} 次): {e}")
            logger.debug(traceback.format_exc())
            if attempt < max_retries:
                wait = 2 ** attempt + random.uniform(0.5, 2.0)
                logger.info(f"等待 {wait:.2f} 秒后重试获取 rest_id ...")
                time.sleep(wait)
            else:
                logger.error(f"多次尝试仍然无法获取 @{user_name} 的 rest_id")
                return None
    return None


def get_user_tweets_page(user_id: str, cursor: Optional[str] = None) -> Dict[str, Any]:
    """获取单页用户推文（按 API 原样返回，一次请求）"""
    conn = http.client.HTTPSConnection(API_HOST, timeout=30)
    headers = {'apikey': API_KEY}
    url = f"/sapi/UserTweets?user_id={user_id}"
    if cursor:
        url += f"&cursor={cursor}"
    else:
        url += "&cursor=null"
    conn.request("GET", url, '', headers)
    res = conn.getresponse()
    data = res.read()
    text = data.decode("utf-8")
    try:
        return json.loads(text)
    except Exception as e:
        # 如果 JSON 解析失败，也打印一点原始内容帮助排查
        logger.error(f"解析 JSON 失败: {e}, 原始响应前 200 字符: {text[:200]}")
        raise


def parse_tweet_datetime(tweet: Dict[str, Any]) -> Optional[datetime]:
    """
    按你给的样例：
    "created_at": "Sat Dec 06 11:50:19 +0000 2025"
    """
    created_at_str = tweet.get("created_at")
    if not created_at_str:
        return None
    try:
        dt = datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.replace(tzinfo=None)
    except Exception as e:
        logger.warning(f"解析 created_at 失败: {created_at_str} ({e})")
        return None


# ==================== 核心：单个用户的窗口测试 ====================

def analyze_single_user_window(user_name: str) -> Dict[str, Any]:
    """
    用你的「耐艹版」逻辑，对单个用户做 UserTweets 翻页：
    - 每页内部多次重试
    - 记录总 tweets 数、最早时间、最晚时间
    返回一个 summary dict
    """
    summary = {
        "user_name": user_name,
        "user_id": None,
        "total_tweets": 0,            # 本次抓到的条数
        "earliest_dt": None,          # datetime
        "latest_dt": None,            # datetime
        "earliest_dt_str": None,
        "latest_dt_str": None,
        "covers_2024_01_or_earlier": False,
        "pages_reached": 0,
        "remark": ""
    }

    logger.info("=" * 60)
    logger.info(f"开始处理用户: @{user_name}")
    logger.info("=" * 60)

    user_id = get_user_rest_id(user_name)
    if not user_id:
        summary["remark"] = "rest_id 获取失败"
        return summary

    summary["user_id"] = user_id

    all_tweets_count = 0
    cursor = None
    page = 1
    min_dt = None
    max_dt = None

    while page <= MAX_PAGES_HARD:
        logger.info(f"=== @{user_name} 第 {page} 页 ===")

        tweets = None
        resp = None

        # ---------- 这一页内部重试 ----------
        for attempt in range(1, MAX_RETRIES_PER_PAGE + 1):
            try:
                logger.info(f"请求第 {page} 页，第 {attempt} 次尝试...")
                resp = get_user_tweets_page(user_id, cursor)
                tweets = resp.get("tweets")

                # 明确区分三种情况：
                # 1) tweets 字段不存在 -> 结构异常，重试
                # 2) tweets 为空列表 -> 可能是限流 / 到头，先重试几次再认定
                # 3) tweets 有内容 -> 正常
                if tweets is None:
                    logger.warning(
                        f"第 {page} 页，第 {attempt} 次：响应中没有 'tweets' 字段，准备重试..."
                    )
                elif len(tweets) == 0:
                    logger.warning(
                        f"第 {page} 页，第 {attempt} 次：tweets 数量为 0，"
                        "可能触发限流 / 窗口尽头，准备重试..."
                    )
                else:
                    logger.info(
                        f"第 {page} 页，第 {attempt} 次：成功拿到 {len(tweets)} 条 tweets"
                    )
                    break

            except Exception as e:
                logger.error(
                    f"第 {page} 页，第 {attempt} 次请求出错：{e}，准备稍后重试..."
                )
                logger.debug(traceback.format_exc())

            # 只有失败才 sleep
            sleep_sec = BASE_SLEEP_BETWEEN_PAGES * attempt + random.uniform(0, EXTRA_JITTER)
            logger.info(f"第 {page} 页，第 {attempt} 次失败后，睡眠 {sleep_sec:.2f} 秒再试")
            time.sleep(sleep_sec)

        # ---------- 重试结束后的判断 ----------
        if not tweets:
            # 多次尝试依然拿不到任何 tweets，就认为：要么被限流，要么窗口尽头
            logger.info(
                f"第 {page} 页在重试 {MAX_RETRIES_PER_PAGE} 次后仍然没有有效 tweets，"
                "认为已经达到该用户在当前状态下的可见历史尽头 / 或严重限流，停止。"
            )
            break

        # 正常处理这一页
        logger.info(f"第 {page} 页最终拿到 tweets 数量：{len(tweets)}")
        summary["pages_reached"] = page

        for tw in tweets:
            dt = parse_tweet_datetime(tw)
            if dt is None:
                continue

            if (min_dt is None) or (dt < min_dt):
                min_dt = dt
            if (max_dt is None) or (dt > max_dt):
                max_dt = dt

        all_tweets_count += len(tweets)

        logger.info(f"当前已统计 tweets 数：{all_tweets_count}")
        logger.info(f"当前时间范围大致为：[{min_dt}  ~  {max_dt}]")

        # 处理 next_cursor
        next_cursor = resp.get("next_cursor_str") if isinstance(resp, dict) else None
        if not next_cursor:
            logger.info("@%s 没有 next_cursor 了，API 不再给更多历史。" % user_name)
            break

        cursor = next_cursor
        page += 1

        # 页与页之间也加一点延迟 + 抖动
        sleep_sec = BASE_SLEEP_BETWEEN_PAGES + random.uniform(0, EXTRA_JITTER)
        logger.info(f"页与页之间睡眠 {sleep_sec:.2f} 秒，防止请求过于频繁")
        time.sleep(sleep_sec)

    # ---------- 汇总结果 ----------
    summary["total_tweets"] = all_tweets_count
    summary["earliest_dt"] = min_dt
    summary["latest_dt"] = max_dt

    if min_dt:
        summary["earliest_dt_str"] = min_dt.strftime("%Y-%m-%d %H:%M:%S")
        summary["covers_2024_01_or_earlier"] = (min_dt <= COVER_TARGET_DATE)
    if max_dt:
        summary["latest_dt_str"] = max_dt.strftime("%Y-%m-%d %H:%M:%S")

    if all_tweets_count == 0:
        summary["remark"] = "未获取到任何 tweet（多次重试后仍为空）"
    else:
        if summary["covers_2024_01_or_earlier"]:
            summary["remark"] = "最早时间 ≤ 2024-01-01 ✅"
        else:
            summary["remark"] = "最早时间 > 2024-01-01 ❌"

    logger.info(
        f"@{user_name} 统计完成：total={all_tweets_count}, "
        f"earliest={summary['earliest_dt_str']}, latest={summary['latest_dt_str']}, "
        f"covers_2024_01={summary['covers_2024_01_or_earlier']}, pages={summary['pages_reached']}"
    )
    logger.info("=" * 60 + "\n")

    return summary


# ==================== 主函数：批量跑 a_kol ====================

def main():
    logger.info("读取 a_kol.xlsx 文件...")
    df_kol = pd.read_excel(KOL_EXCEL_PATH)
    handles_raw = df_kol["Twitter Handle"].tolist()
    # 去掉 @ 并 strip
    handles = [str(h).lstrip("@").strip() for h in handles_raw]

    logger.info(f"共读取 {len(handles)} 个账号")

    summaries: List[Dict[str, Any]] = []
    start_all = time.time()

    # 你也可以先只测前几个账号：
    # handles_to_test = handles[:5]
    handles_to_test = handles

    for i, name in enumerate(handles_to_test, 1):
        logger.info(f"整体进度：{i}/{len(handles_to_test)} (@{name})")
        try:
            summary = analyze_single_user_window(name)
        except Exception as e:
            logger.error(f"处理 @{name} 时发生未捕获异常: {e}")
            logger.error(traceback.format_exc())
            summary = {
                "user_name": name,
                "user_id": None,
                "total_tweets": 0,
                "earliest_dt": None,
                "latest_dt": None,
                "earliest_dt_str": None,
                "latest_dt_str": None,
                "covers_2024_01_or_earlier": False,
                "pages_reached": 0,
                "remark": f"未捕获异常: {e}"
            }
        summaries.append(summary)

        # 每个用户之间再稍微休息一下，避免太猛
        time.sleep(1.0)

    elapsed = time.time() - start_all
    logger.info(f"所有账号处理完成，总耗时 {elapsed:.2f} 秒")

    # 汇总成 DataFrame 输出
    df_summary = pd.DataFrame(summaries)
    # 不需要保存 datetime 对象本身，只保留 *_str 即可
    cols_order = [
        "user_name", "user_id", "total_tweets",
        "earliest_dt_str", "latest_dt_str",
        "covers_2024_01_or_earlier", "pages_reached", "remark"
    ]
    df_summary = df_summary[cols_order]

    df_summary.to_excel(OUT_SUMMARY_PATH, index=False)
    logger.info(f"汇总结果已保存到：{OUT_SUMMARY_PATH}")


if __name__ == "__main__":
    main()


In [1]:
import snscrape.modules.twitter as sntwitter
import pandas as pd
query = "AcmeCorp since:2025-10-01 until:2025-10-31"
tweets_list = []
for i, tweet in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
    tweets_list.append([
        tweet.id, tweet.date, tweet.user.username, tweet.content,
        tweet.replyCount, tweet.retweetCount, tweet.likeCount
    ])
    if i >= 999:  # 限制为1000条
        break
df = pd.DataFrame(tweets_list, columns=[
    "TweetID", "Date", "Username", "Text", "Replies", "Retweets", "Likes"
])
print(df.head())

AttributeError: 'FileFinder' object has no attribute 'find_module'

In [11]:
pip install snscrape

Note: you may need to restart the kernel to use updated packages.
